# Neural Networks with Keras — A Complete Beginner's Guide

*Consolidated from the DTSC301 course (Weeks 1–10) into a single, run-it-top-to-bottom notebook.*

This notebook is built for someone who has **never trained a neural network before**. By the end you'll be comfortable with three things:

1. **Keras** — how to build, train, and evaluate a model in a few lines of code.
2. **Data** — how to load it, shape it, scale it, and split it *without cheating*.
3. **Neural networks** — what they are, the main **use cases** (classification, regression, images, sequences, text), and how to read the **metrics** that tell you whether a model is any good.

### How to use this notebook
- **Run the cells in order, top to bottom.** Each section builds on the last, but every major section reloads its own data so you can also jump around.
- **Read the text, then run the code.** The code is short on purpose — the *ideas* are the point.
- **Epoch counts are kept small** so everything runs quickly. Wherever you see a small `epochs=`, feel free to turn it up once you understand what's happening.
- **Best run in [Google Colab](https://colab.research.google.com)** (free, TensorFlow pre-installed). Local also works if you have TensorFlow installed.

> 🧭 **A note on the datasets.** The original course used some datasets that live in external files (a cats-vs-dogs folder, a traffic CSV, downloaded word vectors). To make this notebook run anywhere with no missing files, we've swapped those for datasets that **ship with Keras** or that we **generate on the fly**. The *techniques* are identical — only the data differs.

### The map of where we're going
| Part | You'll learn | Use case |
|---|---|---|
| 1 | What a neural net actually is | (intuition) |
| 2 | Your first Keras model | image classification |
| 3 | Working with data properly | tabular / regression |
| 4 | The 3 core problem types + which loss & metric to use | binary, multi-class, regression |
| 5 | Reading metrics like a pro | evaluation |
| 6 | Overfitting and how to beat it | regularization |
| 7 | Choosing hyperparameters | tuning |
| 8 | Convolutional networks | images |
| 9 | Transfer learning | reuse big models |
| 10 | Recurrent networks | sequences / time series |
| 11 | Embeddings | text |
| 12 | Cheat sheets & where to go next | (reference) |

---
## Part 0 — Setup

A neural network in Keras needs surprisingly little scaffolding. Let's get imports out of the way once.

- **TensorFlow** is the engine that does the actual math (and uses your GPU if you have one).
- **Keras** is the friendly API on top of TensorFlow — it's what we actually type.
- **NumPy** handles arrays of numbers; **Matplotlib** draws graphs.

Run this cell first. In Colab everything is already installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Make results roughly reproducible from run to run
keras.utils.set_random_seed(42)

print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print(
      'GPU:',
      'available'
      if tf.config.list_physical_devices('GPU')
      else 'none (CPU is fine here)',
)

### A helper we'll reuse: plotting the training history

Every `model.fit(...)` hands back a **history** object — a record of the loss and metrics at every epoch. Watching those curves is *the* core skill for judging whether training is going well, so let's write one small function now and reuse it throughout.

In [ ]:
def plot_history(history, metric='accuracy'):
      # Plot training vs validation loss (left) and one metric (right) across epochs.
      h = history.history
      epochs = range(1, len(h['loss']) + 1)
      fig, axes = plt.subplots(1, 2, figsize=(11, 4))

      axes[0].plot(epochs, h['loss'], 'o-', label='Training loss')
      if 'val_loss' in h:
            axes[0].plot(epochs, h['val_loss'], 'o-', label='Validation loss')
      axes[0].set_title('Loss')
      axes[0].set_xlabel('Epoch')
      axes[0].legend()

      if metric in h:
            axes[1].plot(epochs, h[metric], 'o-', label='Training ' + metric)
            if ('val_' + metric) in h:
                  axes[1].plot(
                        epochs, h['val_' + metric], 'o-', label='Validation ' + metric
                  )
            axes[1].set_title(metric.capitalize())
            axes[1].set_xlabel('Epoch')
            axes[1].legend()
      else:
            axes[1].axis('off')

      plt.tight_layout()
      plt.show()


print('plot_history() ready.')

---
## Part 1 — What *is* a neural network? (built by hand)

Before we let Keras do the work, let's demystify the whole thing with about 15 lines of plain Python. The "learning" in machine learning is just **repeatedly nudging some numbers in the direction that reduces error**. That's the whole trick.

Say we want to predict `y` from `x` with the simplest possible model — a straight line:

$$\hat{y} = b + w \cdot x$$

- `w` (weight) and `b` (bias) are the **parameters** — the numbers the model learns.
- We measure how wrong we are with a **loss** (here, squared error).
- **Gradient descent** tells us which way to move `w` and `b` to shrink the loss; we take a small step that way and repeat.

A neural network is this same idea, stacked and scaled up massively. Watch the loss fall:

In [ ]:
# Some tiny made-up data
x = np.array([140, 70, 50, 163], dtype=float)
y = np.array([106, 80, 83, 43], dtype=float)

b, w = 80.0, 1.2  # start the parameters at a guess
learning_rate = 0.0001  # how big a step we take each time

for step in range(30):
      # 1) FORWARD PASS: predict, then measure the error
      y_pred = b + w * x
      loss = np.mean(0.5 * (y_pred - y) ** 2)

      # 2) BACKWARD PASS: how does the loss change w.r.t. b and w? (calculus, by hand)
      dloss_dpred = y_pred - y
      grad_b = np.mean(dloss_dpred * 1)  # dpred/db = 1
      grad_w = np.mean(dloss_dpred * x)  # dpred/dw = x

      # 3) UPDATE: nudge the parameters downhill
      b -= learning_rate * grad_b
      w -= learning_rate * grad_w

      if step % 5 == 0:
            print(f'step {step:2d}  loss={loss:8.2f}  b={b:6.2f}  w={w:5.3f}')

print(f'\nLearned line:  y = {b:.2f} + {w:.3f} * x')

That loop **is** training. A real neural network differs in only two ways: it has *many* weights (thousands to billions), and instead of a straight line it stacks **layers** with **activation functions** so it can bend to fit curves and complex patterns. Keras does the forward pass, the calculus, and the updates for you — but the loop above is what happens under the hood every time you call `.fit()`.

**Vocabulary you just met**
- **Parameters / weights** — the numbers the model learns (`w`, `b`).
- **Loss** — one number measuring how wrong the model is. Training = making it smaller.
- **Gradient descent** — nudging parameters in the direction that lowers the loss.
- **Learning rate** — the size of each nudge. Too big overshoots; too small crawls.
- **Epoch** — one full pass over all the training data.

---
## Part 2 — Your first Keras model

Time to let Keras do the heavy lifting. The classic "hello world" of deep learning is **MNIST**: 70,000 tiny greyscale images of handwritten digits (0–9). Our job: look at a 28×28 image and predict which digit it is. This is a **multi-class classification** problem (10 possible classes).

MNIST ships with Keras, so loading it is one line. The first run downloads ~11 MB.

In [ ]:
from tensorflow.keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

print(
      'Training images:', train_images.shape
)  # (60000, 28, 28)  -> 60k images, each 28x28
print('Training labels:', train_labels.shape)  # (60000,)
print('First 10 labels:', train_labels[:10])

Let's actually *look* at the data — never train on data you haven't eyeballed.

In [ ]:
plt.figure(figsize=(8, 2))
for i in range(6):
      plt.subplot(1, 6, i + 1)
      plt.imshow(train_images[i], cmap='gray')
      plt.title(f'label: {train_labels[i]}')
      plt.axis('off')
plt.tight_layout()
plt.show()

### Preparing the data
Two small prep steps, and *why* each one matters:

1. **Reshape + scale the images.** A basic dense network wants a flat vector, not a 2-D grid, so we flatten each 28×28 image into 784 numbers. We also divide by 255 so every pixel is between 0 and 1 — networks train far more happily on small, similarly-scaled inputs.
2. **One-hot encode the labels.** The digit `3` becomes `[0,0,0,1,0,0,0,0,0,0]`. This matches the 10-number output our network will produce (one score per digit).

In [ ]:
# 1) flatten 28x28 -> 784, and scale pixels to 0..1
x_train = train_images.reshape((60000, 28 * 28)).astype('float32') / 255
x_test = test_images.reshape((10000, 28 * 28)).astype('float32') / 255

# 2) one-hot encode the labels
y_train = keras.utils.to_categorical(train_labels)
y_test = keras.utils.to_categorical(test_labels)

print('x_train:', x_train.shape, '| y_train:', y_train.shape)
print('label 3 becomes:', y_train[np.where(train_labels == 3)[0][0]])

### Building the network
Here's the whole model. Three ideas to notice:

- `Sequential` just means "a straight stack of layers, one after another."
- Each `Dense` layer is fully connected; the number (512, 10) is how many **neurons** it has.
- `activation="relu"` lets the network model non-linear patterns. The final `activation="softmax"` turns the 10 outputs into probabilities that add up to 1 — perfect for "pick one of 10 classes."

In [ ]:
model = keras.Sequential(
      [
            layers.Input(shape=(28 * 28,)),  # 784 inputs, one per pixel
            layers.Dense(512, activation='relu'),  # hidden layer
            layers.Dense(10, activation='softmax'),  # 10 outputs = 10 digit classes
      ]
)

model.summary()

### Compiling: choosing how the model learns
`compile()` wires up three things (we'll unpack the choices in Part 4):
- **optimizer** — the flavour of gradient descent (`"rmsprop"` is a solid default).
- **loss** — what training tries to minimise. `categorical_crossentropy` is the standard for one-hot multi-class.
- **metrics** — human-friendly scores to watch. `accuracy` = fraction of digits classified correctly.

In [ ]:
model.compile(
      optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy']
)

### Training
`fit()` runs the learning loop from Part 1 for us. `epochs=5` means five passes over the data; `batch_size=128` means it updates the weights after every 128 images.

In [ ]:
history = model.fit(
      x_train,
      y_train,
      epochs=5,
      batch_size=128,
      validation_split=0.1,  # hold back 10% to watch for overfitting
      verbose=2,
)

### Evaluating on data the model has never seen
The real test is the **test set** — images the model didn't learn from. `evaluate()` returns the loss and accuracy.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.3f}')

plot_history(history, metric='accuracy')

### Making predictions
A trained model outputs 10 probabilities per image. `argmax` picks the highest one.

In [ ]:
probs = model.predict(x_test[:5], verbose=0)
predicted = probs.argmax(axis=1)
print('Predicted:', predicted)
print('Actual:   ', test_labels[:5])

### Saving and reloading a model
Once you've trained something good, save it so you never have to retrain. The modern Keras format is `.keras`.

In [ ]:
model.save('my_first_model.keras')
reloaded = keras.models.load_model('my_first_model.keras')
print(
      'Reloaded OK — test accuracy:',
      round(reloaded.evaluate(x_test, y_test, verbose=0)[1], 3),
)

🎉 **You just trained a neural network.** Load data → look at it → prep it → build → compile → fit → evaluate. **That six-step rhythm is the whole game** — every model in this notebook follows it. The rest is learning which knobs to turn for different problems.

---
## Part 3 — Working with data properly

Models are easy; **data is where beginners lose accuracy without realising it.** This part covers the handful of data habits that separate a working model from a misleading one. We'll switch to a **tabular regression** problem: predicting California house prices from features like income and location.

We use `scikit-learn`'s California housing dataset (a clean, modern replacement for the old Boston dataset). It's numbers in a table — the format most real-world business data arrives in.

In [ ]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
X, y = data.data, data.target  # X: features (20640, 8), y: median house value ($100k)

print('Feature names:', data.feature_names)
print('X shape:', X.shape, '| y shape:', y.shape)
print('First row of features:', np.round(X[0], 2))
print('Its target (house value in $100k):', y[0])

### Habit 1 — Split into train / validation / test *before* touching the data
Three separate piles, each with a job:
- **Training set** — the model learns from this.
- **Validation set** — you peek at this while tuning, to catch overfitting. The model never learns from it directly.
- **Test set** — locked in a drawer until the very end. It's your one honest estimate of real-world performance.

The cardinal sin is letting test information leak into training. Split first, then do everything else.

In [ ]:
n = len(X)
idx = np.random.permutation(n)  # shuffle the row order
n_test, n_val = int(0.15 * n), int(0.15 * n)

test_idx = idx[:n_test]
val_idx = idx[n_test : n_test + n_val]
train_idx = idx[n_test + n_val :]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f'train: {len(X_train)}  |  val: {len(X_val)}  |  test: {len(X_test)}')

### Habit 2 — Standardise features, using training statistics only
The 8 features live on wildly different scales (income vs. number of rooms vs. latitude). Neural networks train much better when every feature is centred at 0 with a spread of about 1. So we subtract the mean and divide by the standard deviation.

**The subtle, critical bit:** compute the mean and standard deviation **from the training set only**, then apply those same numbers to validation and test. If you standardise using the whole dataset, information from the test set leaks into training — this is **data leakage**, and it makes your model look better than it really is.

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

# apply the TRAINING stats to all three sets
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print('After standardising, training means are ~0:', np.round(X_train.mean(axis=0), 2))
print('...and training std devs are ~1:', np.round(X_train.std(axis=0), 2))

### Build a regression model
Regression predicts a **number**, so two things change compared to the digit classifier:
- The final layer has **1 neuron** with **no activation** (a plain linear output — it can be any value).
- The loss is **mean squared error (`mse`)**, and we watch **mean absolute error (`mae`)**, which is in the same units as the target (here, $100k).

In [ ]:
def build_regression_model():
      m = keras.Sequential(
            [
                  layers.Input(shape=(X_train.shape[1],)),
                  layers.Dense(64, activation='relu'),
                  layers.Dense(64, activation='relu'),
                  layers.Dense(1),  # 1 output, no activation -> regression
            ]
      )
      m.compile(optimizer='adam', loss='mse', metrics=['mae'])
      return m


reg_model = build_regression_model()
history = reg_model.fit(
      X_train,
      y_train,
      validation_data=(X_val, y_val),
      epochs=30,
      batch_size=128,
      verbose=0,
)

plot_history(history, metric='mae')
test_mse, test_mae = reg_model.evaluate(X_test, y_test, verbose=0)
print(
      f'Test MAE: {test_mae:.3f}  (i.e. off by about ${test_mae * 100:.0f}k on average)'
)

### Habit 3 — When data is scarce, use K-fold cross-validation
With a small dataset, a single validation split is noisy — you might get lucky or unlucky. **K-fold** splits the data into K parts, trains K times (each time holding out a different part for validation), and averages the scores. It's slower but gives a far more trustworthy estimate. Here's the idea on 4 folds:

In [ ]:
k = 4
fold_size = len(X_train) // k
scores = []

for fold in range(k):
      # carve out this fold as validation, the rest as training
      v_start, v_end = fold * fold_size, (fold + 1) * fold_size
      Xv, yv = X_train[v_start:v_end], y_train[v_start:v_end]
      Xt = np.concatenate([X_train[:v_start], X_train[v_end:]], axis=0)
      yt = np.concatenate([y_train[:v_start], y_train[v_end:]], axis=0)

      m = build_regression_model()
      m.fit(Xt, yt, epochs=20, batch_size=128, verbose=0)
      scores.append(m.evaluate(Xv, yv, verbose=0)[1])  # mae

print('MAE per fold:', np.round(scores, 3))
print(f'Average MAE: {np.mean(scores):.3f}  (+/- {np.std(scores):.3f})')

**Recap of Part 3 — the four data commandments:**
1. Split into train / validation / test *first*.
2. Standardise features using **training** statistics only (avoid leakage).
3. Match your output layer and loss to the task (1 linear neuron + `mse` for regression).
4. When data is scarce, trust K-fold over a single split.

---
## Part 4 — The three core problem types (and which loss & metric to use)

Almost every "predict a value" task is one of three shapes. Getting the **output layer**, **loss**, and **metric** right for the shape is the single most important modelling decision — and it's mechanical once you know the pattern.

| Problem | Example | Last layer | Loss | Watch this metric |
|---|---|---|---|---|
| **Binary classification** | spam / not spam | `Dense(1, "sigmoid")` | `binary_crossentropy` | accuracy, precision/recall, AUC |
| **Multi-class classification** | which of 10 digits | `Dense(C, "softmax")` | `categorical_crossentropy` | accuracy |
| **Regression** | house price | `Dense(1)` (no activation) | `mse` | MAE |

We already did multi-class (MNIST) and regression (housing). Let's nail **binary classification**, then compare all three side by side.

### Binary classification: movie-review sentiment (IMDB)
The IMDB dataset is 50,000 movie reviews labelled positive (1) or negative (0). It ships with Keras, already converted to sequences of word-index numbers. We keep only the 10,000 most common words.

In [ ]:
from tensorflow.keras.datasets import imdb

NUM_WORDS = 10000
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(
      num_words=NUM_WORDS
)

print('Number of training reviews:', len(train_data))
print('First review is a list of word indices:', train_data[0][:12], '...')
print('Its label (1=positive, 0=negative):', train_labels[0])

A network needs fixed-size numeric input, but reviews vary in length. A simple, effective trick: **multi-hot encoding** — turn each review into a 10,000-long vector that's 1 for every word present and 0 elsewhere. (In Part 11 we'll meet a smarter approach: embeddings.)

In [ ]:
def multi_hot(sequences, dim=NUM_WORDS):
      out = np.zeros((len(sequences), dim), dtype='float32')
      for i, seq in enumerate(sequences):
            out[i, seq] = 1.0  # mark every word that appears
      return out


x_train = multi_hot(train_data)
x_test = multi_hot(test_data)
y_train = np.asarray(train_labels, dtype='float32')
y_test = np.asarray(test_labels, dtype='float32')

print('x_train shape:', x_train.shape)  # (25000, 10000)

The model is small. The star is the **last layer**: one neuron with a **sigmoid**, which squashes the output to a probability between 0 and 1 ("how positive is this review?"). Paired with **binary_crossentropy** loss, that's the canonical binary-classification setup.

In [ ]:
bin_model = keras.Sequential(
      [
            layers.Input(shape=(NUM_WORDS,)),
            layers.Dense(16, activation='relu'),
            layers.Dense(16, activation='relu'),
            layers.Dense(1, activation='sigmoid'),  # <-- binary head
      ]
)
bin_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

history = bin_model.fit(
      x_train, y_train, epochs=10, batch_size=512, validation_split=0.2, verbose=0
)
plot_history(history, metric='accuracy')

Notice the shape of those curves: training keeps improving, but **validation loss bottoms out and then climbs**. That upward turn is **overfitting** — the model is memorising the training reviews instead of learning general sentiment. Part 6 is entirely about fixing it. For now, the fix is simple: stop training around the epoch where validation loss was lowest.

In [ ]:
# Retrain for just the good number of epochs, then judge on the untouched test set
bin_model = keras.Sequential(
      [
            layers.Input(shape=(NUM_WORDS,)),
            layers.Dense(16, activation='relu'),
            layers.Dense(16, activation='relu'),
            layers.Dense(1, activation='sigmoid'),
      ]
)
bin_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
bin_model.fit(x_train, y_train, epochs=4, batch_size=512, verbose=0)

test_loss, test_acc = bin_model.evaluate(x_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.3f}')

### Why the head/loss/metric must match the task
It's worth seeing *why* the table isn't arbitrary:
- **Sigmoid** gives one probability → great for yes/no. **Softmax** gives a probability *distribution* over classes → great for pick-one-of-many. A linear output can be any number → great for regression.
- The **loss** must be differentiable and aligned with the goal. Cross-entropy punishes confident-and-wrong probability predictions; MSE punishes being numerically far off.
- The **metric** is for *humans*. Accuracy is intuitive for classification; MAE ("off by about $X") is intuitive for regression. The loss is for the optimiser; the metric is for you.

Get these three right and you're 80% of the way to a working model on any new problem.

---
## Part 5 — Reading metrics like a pro

Accuracy is comforting but often **lies**. If 99% of transactions are legitimate, a model that blindly predicts "legit" every time scores 99% accuracy while catching *zero* fraud. To really know a classifier, you need the metrics below. We'll use the IMDB sentiment model from Part 4.

In [ ]:
# Get the model's predicted probabilities, then turn them into 0/1 decisions at threshold 0.5
probs = bin_model.predict(x_test, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)
print('First 8 probabilities:', np.round(probs[:8], 2))
print('First 8 decisions:    ', preds[:8])

### The confusion matrix — the source of all classification metrics
Every prediction falls into one of four buckets. Everything else is derived from these counts.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, preds)
ConfusionMatrixDisplay(cm, display_labels=['negative', 'positive']).plot(cmap='Blues')
plt.title('Confusion matrix')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives : {tn}   (said negative, was negative)')
print(f'False Positives: {fp}   (said positive, was negative)  <- false alarm')
print(f'False Negatives: {fn}   (said negative, was positive)  <- missed it')
print(f'True Positives : {tp}   (said positive, was positive)')

### Precision, recall, F1 — and why you can't maximise all of them
- **Precision** = of everything I *flagged* positive, how much really was? (Punishes false alarms.)
- **Recall** = of everything that *was* positive, how much did I catch? (Punishes misses.)
- **F1** = the harmonic mean of the two — one number when you want a single summary.

There's a **trade-off**: lower the decision threshold and you catch more positives (higher recall) but raise false alarms (lower precision). The "right" balance depends on your business cost. Missing cancer (false negative) is far worse than a false alarm; flooding users with spam filters that block real mail is the opposite.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, preds, target_names=['negative', 'positive']))

### ROC / AUC — quality across *all* thresholds
Precision and recall depend on where you set the 0.5 threshold. **AUC** (Area Under the ROC Curve) summarises performance across *every* threshold in a single number: 1.0 is perfect, 0.5 is a coin flip. It's the go-to metric for ranking-style and imbalanced problems.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, _ = roc_curve(y_test, probs)
auc = roc_auc_score(y_test, probs)

plt.plot(fpr, tpr, label=f'model (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='coin flip (AUC = 0.5)')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend()
plt.show()

### Regression metrics
Classification metrics don't apply to numbers. For regression you mostly use:
- **MAE** (mean absolute error) — average size of the miss, in the target's own units. Easy to explain.
- **MSE / RMSE** — squares the errors, so big misses hurt more. RMSE is back in the original units.
- **MAPE** — error as a *percentage*, handy when scale varies.
- **R²** — fraction of variance explained (1.0 = perfect, 0 = no better than guessing the mean).

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# reload+standardise a quick housing model prediction (self-contained mini-repeat)
from sklearn.datasets import fetch_california_housing

d = fetch_california_housing()
Xh, yh = d.data, d.target
cut = int(0.85 * len(Xh))
Xtr, Xte, ytr, yte = Xh[:cut], Xh[cut:], yh[:cut], yh[cut:]
m_, s_ = Xtr.mean(0), Xtr.std(0)
Xtr, Xte = (Xtr - m_) / s_, (Xte - m_) / s_
quick = keras.Sequential(
      [layers.Input(shape=(8,)), layers.Dense(64, activation='relu'), layers.Dense(1)]
)
quick.compile(optimizer='adam', loss='mse')
quick.fit(Xtr, ytr, epochs=20, batch_size=128, verbose=0)
yhat = quick.predict(Xte, verbose=0).ravel()

print(f'MAE : {mean_absolute_error(yte, yhat):.3f}')
print(f'RMSE: {np.sqrt(mean_squared_error(yte, yhat)):.3f}')
print(f'R^2 : {r2_score(yte, yhat):.3f}')

**The metric mindset:** the loss is what the optimiser minimises; the **metric is how you decide if the model is good enough for its real job.** Always pick the metric that reflects the actual cost of being wrong — not just whichever number looks highest.

---
## Part 6 — Overfitting, and how to beat it

**Overfitting** is the central struggle of machine learning: a model that has *memorised* the training data instead of *learning the pattern*. You spot it when training loss keeps dropping while validation loss turns and climbs (you saw this in Part 4).

The cure is **regularization** — anything that makes it harder for the model to memorise. Four tools, from most to least common. We'll demonstrate on the IMDB data from Part 4.

In [ ]:
# Rebuild the IMDB data cleanly for this section
from tensorflow.keras.datasets import imdb

NUM_WORDS = 10000
(tr, tr_y), (te, te_y) = imdb.load_data(num_words=NUM_WORDS)


def multi_hot(seqs, dim=NUM_WORDS):
      out = np.zeros((len(seqs), dim), dtype='float32')
      for i, s in enumerate(seqs):
            out[i, s] = 1.0
      return out


x_tr, x_te = multi_hot(tr), multi_hot(te)
y_tr, y_te = np.asarray(tr_y, 'float32'), np.asarray(te_y, 'float32')


def make_and_fit(model, epochs=20):
      model.compile(
            optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy']
      )
      return model.fit(
            x_tr, y_tr, epochs=epochs, batch_size=512, validation_split=0.4, verbose=0
      )


print('IMDB reloaded for the overfitting demos.')

### Tool 0 — A baseline that overfits
First, a deliberately over-powered model so we have something to fix.

In [ ]:
baseline = keras.Sequential(
      [
            layers.Input(shape=(NUM_WORDS,)),
            layers.Dense(512, activation='relu'),
            layers.Dense(512, activation='relu'),
            layers.Dense(1, activation='sigmoid'),
      ]
)
h_base = make_and_fit(baseline)
plot_history(h_base)

### Tool 1 — Use a smaller model (less capacity)
The simplest fix: a model with fewer neurons has less room to memorise. Capacity is a dial — too much overfits, too little underfits.

In [ ]:
smaller = keras.Sequential(
      [
            layers.Input(shape=(NUM_WORDS,)),
            layers.Dense(16, activation='relu'),
            layers.Dense(16, activation='relu'),
            layers.Dense(1, activation='sigmoid'),
      ]
)
h_small = make_and_fit(smaller)

# compare validation loss curves
plt.plot(h_base.history['val_loss'], 'o-', label='big model')
plt.plot(h_small.history['val_loss'], 'o-', label='small model')
plt.xlabel('epoch')
plt.ylabel('validation loss')
plt.legend()
plt.title('Smaller model overfits less')
plt.show()

### Tool 2 — Weight regularization (L2)
Penalise large weights by adding them to the loss. The model is pushed toward *simpler* solutions. `l2(0.001)` is a typical starting strength.

In [ ]:
from tensorflow.keras import regularizers

l2_model = keras.Sequential(
      [
            layers.Input(shape=(NUM_WORDS,)),
            layers.Dense(
                  16, activation='relu', kernel_regularizer=regularizers.l2(0.001)
            ),
            layers.Dense(
                  16, activation='relu', kernel_regularizer=regularizers.l2(0.001)
            ),
            layers.Dense(1, activation='sigmoid'),
      ]
)
h_l2 = make_and_fit(l2_model)

plt.plot(h_base.history['val_loss'], 'o-', label='baseline')
plt.plot(h_l2.history['val_loss'], 'o-', label='L2 regularized')
plt.xlabel('epoch')
plt.ylabel('validation loss')
plt.legend()
plt.show()

### Tool 3 — Dropout
During training, `Dropout(0.5)` randomly zeroes out half the neurons' outputs each step. The network can't rely on any single neuron, so it learns more robust, redundant features. It's one of the most effective tricks in deep learning — and it does nothing at prediction time (Keras handles that switch automatically).

In [ ]:
dropout_model = keras.Sequential(
      [
            layers.Input(shape=(NUM_WORDS,)),
            layers.Dense(16, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(16, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(1, activation='sigmoid'),
      ]
)
h_drop = make_and_fit(dropout_model)

plt.plot(h_base.history['val_loss'], 'o-', label='baseline')
plt.plot(h_drop.history['val_loss'], 'o-', label='with dropout')
plt.xlabel('epoch')
plt.ylabel('validation loss')
plt.legend()
plt.show()

### Tool 4 — Early stopping (via callbacks)
Instead of guessing the right number of epochs, let Keras stop automatically when validation loss stops improving, and roll back to the best weights. **Callbacks** are hooks that run during training; `EarlyStopping` and `ModelCheckpoint` are the two you'll use constantly.

In [ ]:
callbacks = [
      keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=3, restore_best_weights=True
      ),
]

final = keras.Sequential(
      [
            layers.Input(shape=(NUM_WORDS,)),
            layers.Dense(16, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(16, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(1, activation='sigmoid'),
      ]
)
final.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
h_final = final.fit(
      x_tr,
      y_tr,
      epochs=50,
      batch_size=512,
      validation_split=0.4,
      callbacks=callbacks,
      verbose=0,
)

print(f'Stopped after {len(h_final.history["loss"])} epochs (asked for up to 50).')
print(f'Test accuracy: {final.evaluate(x_te, y_te, verbose=0)[1]:.3f}')

**The anti-overfitting toolkit, in the order to reach for it:** get more data if you can → reduce capacity → add dropout and/or L2 → always use early stopping. You'll combine these on nearly every real project.

---
## Part 7 — Choosing hyperparameters (tuning)

**Parameters** are learned by the model (weights). **Hyperparameters** are the choices *you* make before training: how many layers, how many neurons, learning rate, dropout rate, batch size. There's no formula for the best combination — you search.

### Manual tuning: a small grid
The honest starting point is a loop over a few options, judged on the **validation** set (never the test set). We'll keep reusing the IMDB data from Part 6.

In [ ]:
def build(units, dropout, lr):
      m = keras.Sequential(
            [
                  layers.Input(shape=(NUM_WORDS,)),
                  layers.Dense(units, activation='relu'),
                  layers.Dropout(dropout),
                  layers.Dense(1, activation='sigmoid'),
            ]
      )
      m.compile(
            optimizer=keras.optimizers.RMSprop(learning_rate=lr),
            loss='binary_crossentropy',
            metrics=['accuracy'],
      )
      return m


results = []
for units in [16, 64]:
      for dropout in [0.3, 0.6]:
            m = build(units, dropout, lr=1e-3)
            h = m.fit(
                  x_tr, y_tr, epochs=5, batch_size=512, validation_split=0.4, verbose=0
            )
            val_acc = max(h.history['val_accuracy'])
            results.append((units, dropout, val_acc))
            print(
                  f'units={units:3d}  dropout={dropout}  ->  best val_acc={val_acc:.3f}'
            )

best = max(results, key=lambda r: r[2])
print(f'\nBest combo: units={best[0]}, dropout={best[1]}  (val_acc={best[2]:.3f})')

### Automated tuning: KerasTuner
For bigger searches, **KerasTuner** explores the space intelligently. You describe the *range* of each hyperparameter and a strategy (here, `RandomSearch`) does the rest. It isn't pre-installed, so we `pip install` it first. *(This cell is a little slower; reduce `max_trials` if you're impatient.)*

In [ ]:
try:
      import keras_tuner as kt
except ImportError:
      import subprocess, sys

      subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'keras-tuner'])
      import keras_tuner as kt


def model_builder(hp):
      units = hp.Int('units', min_value=16, max_value=128, step=32)
      dropout = hp.Float('dropout', 0.2, 0.6, step=0.2)
      lr = hp.Choice('lr', [1e-2, 1e-3, 1e-4])
      m = keras.Sequential(
            [
                  layers.Input(shape=(NUM_WORDS,)),
                  layers.Dense(units, activation='relu'),
                  layers.Dropout(dropout),
                  layers.Dense(1, activation='sigmoid'),
            ]
      )
      m.compile(
            optimizer=keras.optimizers.RMSprop(lr),
            loss='binary_crossentropy',
            metrics=['accuracy'],
      )
      return m


tuner = kt.RandomSearch(
      model_builder,
      objective='val_accuracy',
      max_trials=4,
      overwrite=True,
      directory='kt_dir',
      project_name='imdb',
)
tuner.search(x_tr, y_tr, epochs=3, batch_size=512, validation_split=0.4, verbose=0)

best_hp = tuner.get_best_hyperparameters(1)[0]
print('Best hyperparameters found:')
print('  units  :', best_hp.get('units'))
print('  dropout:', best_hp.get('dropout'))
print('  lr     :', best_hp.get('lr'))

**Tuning discipline:** tune on validation data, change one thing at a time when doing it by hand, and touch the test set only once at the very end. Sensible defaults to start from: `adam` optimizer, learning rate `1e-3`, a couple of hidden layers, dropout `0.3–0.5`.

---
## Part 8 — Convolutional networks for images

Flattening an image into a vector (like we did with MNIST) throws away its structure — which pixels are *next to* which. **Convolutional Neural Networks (CNNs)** keep the 2-D grid and scan small **filters** across it to detect local patterns: edges, then textures, then shapes, then objects. They are the workhorse of computer vision.

Two new layer types:
- **`Conv2D`** slides learnable filters over the image, producing "feature maps" that light up where a pattern is found.
- **`MaxPooling2D`** shrinks the maps by keeping the strongest signal in each little region — cheaper, and more robust to small shifts.

We'll use **Fashion-MNIST** (28×28 greyscale clothing images, 10 classes) — bundled with Keras, and more interesting than digits.

In [ ]:
from tensorflow.keras.datasets import fashion_mnist

(Xtr, ytr), (Xte, yte) = fashion_mnist.load_data()
class_names = [
      'T-shirt',
      'Trouser',
      'Pullover',
      'Dress',
      'Coat',
      'Sandal',
      'Shirt',
      'Sneaker',
      'Bag',
      'Ankle boot',
]

# CNNs expect a channel dimension: (height, width, channels)
Xtr = Xtr.reshape(-1, 28, 28, 1).astype('float32') / 255
Xte = Xte.reshape(-1, 28, 28, 1).astype('float32') / 255

plt.figure(figsize=(9, 2))
for i in range(6):
      plt.subplot(1, 6, i + 1)
      plt.imshow(Xtr[i].squeeze(), cmap='gray')
      plt.title(class_names[ytr[i]], fontsize=8)
      plt.axis('off')
plt.tight_layout()
plt.show()

A typical CNN stacks Conv+Pool blocks to extract features, then `Flatten`s and finishes with `Dense` layers to classify. Note the last layer: `Dense(10, "softmax")` — the same multi-class head from Part 2, because this is still a 10-class problem.

In [ ]:
cnn = keras.Sequential(
      [
            layers.Input(shape=(28, 28, 1)),
            layers.Conv2D(32, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.Flatten(),
            layers.Dense(64, activation='relu'),
            layers.Dense(10, activation='softmax'),
      ]
)
cnn.compile(
      optimizer='adam',
      loss='sparse_categorical_crossentropy',  # labels are ints, not one-hot
      metrics=['accuracy'],
)
cnn.summary()

> 💡 **`sparse_categorical_crossentropy`** is a handy shortcut: it's the same loss as `categorical_crossentropy` but lets you pass integer labels (`3`) instead of one-hot vectors (`[0,0,0,1,...]`). One less preprocessing step.

In [ ]:
history = cnn.fit(Xtr, ytr, epochs=5, batch_size=128, validation_split=0.1, verbose=2)
plot_history(history, metric='accuracy')
print(f'Test accuracy: {cnn.evaluate(Xte, yte, verbose=0)[1]:.3f}')

### Data augmentation — free extra training data
A cheap, powerful regularizer for images: randomly flip / rotate / zoom each image a little during training, so the model sees endless variations and generalises better. In modern Keras these are just **layers** you put at the top of the model — active during training, ignored at prediction time.

In [ ]:
augment = keras.Sequential(
      [
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
      ],
      name='augmentation',
)

aug_cnn = keras.Sequential(
      [
            layers.Input(shape=(28, 28, 1)),
            augment,  # <-- augmentation happens here
            layers.Conv2D(32, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Conv2D(64, (3, 3), activation='relu'),
            layers.MaxPooling2D((2, 2)),
            layers.Flatten(),
            layers.Dense(64, activation='relu'),
            layers.Dense(10, activation='softmax'),
      ]
)
aug_cnn.compile(
      optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy']
)
aug_cnn.fit(Xtr, ytr, epochs=5, batch_size=128, validation_split=0.1, verbose=2)
print(f'Augmented test accuracy: {aug_cnn.evaluate(Xte, yte, verbose=0)[1]:.3f}')

CNNs follow the exact same six-step rhythm as every other model — only the *layers* changed. That's the beauty of Keras: once you know the pattern, new problem types are mostly new layer types.

---
## Part 9 — Transfer learning: standing on giants' shoulders

Training a great image model from scratch needs millions of images and serious compute. **Transfer learning** skips that: take a network already trained on ImageNet (1.4M images, 1000 categories), lop off its final classifier, and reuse its rich feature-detectors on *your* problem. It's often the single biggest accuracy win available on small image datasets — and it's how most real-world vision projects actually start.

We'll classify **CIFAR-10** (32×32 colour images, 10 classes) using **MobileNetV2** as a frozen feature extractor. *(This section downloads the CIFAR-10 data and the pretrained weights, so the first run is the slowest in the notebook. We use a subset to keep it snappy.)*

In [ ]:
from tensorflow.keras.datasets import cifar10

(Xtr, ytr), (Xte, yte) = cifar10.load_data()
cifar_names = [
      'airplane',
      'automobile',
      'bird',
      'cat',
      'deer',
      'dog',
      'frog',
      'horse',
      'ship',
      'truck',
]

# Use a subset so this runs in a couple of minutes
Xtr, ytr = Xtr[:6000], ytr[:6000]
Xte, yte = Xte[:2000], yte[:2000]
print('Subset shapes:', Xtr.shape, Xte.shape)

plt.figure(figsize=(9, 2))
for i in range(6):
      plt.subplot(1, 6, i + 1)
      plt.imshow(Xtr[i])
      plt.title(cifar_names[int(ytr[i][0])], fontsize=8)
      plt.axis('off')
plt.tight_layout()
plt.show()

Two prep steps for a pretrained model:
1. **Resize** to what the base network expects (MobileNetV2 likes ≥96×96) and apply its own **`preprocess_input`** scaling.
2. **Freeze** the base (`trainable = False`) so training only updates our small new classifier head — fast, and it won't wreck the pretrained knowledge.

In [ ]:
IMG = 96

base = keras.applications.MobileNetV2(
      input_shape=(IMG, IMG, 3), include_top=False, weights='imagenet'
)
base.trainable = False  # <-- freeze the pretrained features

model = keras.Sequential(
      [
            layers.Input(shape=(32, 32, 3)),
            layers.Resizing(IMG, IMG),  # 32x32 -> 96x96, what MobileNet expects
            layers.Rescaling(
                  1.0 / 127.5, offset=-1
            ),  # scale 0..255 -> -1..1 (MobileNetV2's preprocessing)
            base,  # frozen ImageNet feature extractor
            layers.GlobalAveragePooling2D(),  # squash feature maps to a vector
            layers.Dropout(0.3),
            layers.Dense(10, activation='softmax'),  # our new 10-class head
      ]
)
model.compile(
      optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy']
)
print('Frozen base? ', not base.trainable, '| total layers:', len(model.layers))

In [ ]:
history = model.fit(Xtr, ytr, epochs=5, batch_size=64, validation_split=0.1, verbose=2)
plot_history(history, metric='accuracy')
print(f'Transfer-learning test accuracy: {model.evaluate(Xte, yte, verbose=0)[1]:.3f}')

Reaching solid accuracy on colour photos in a handful of epochs — on a subset, from a laptop — is only possible *because* we reused MobileNetV2's features. The advanced next step is **fine-tuning**: unfreeze the top few layers of the base and train them at a tiny learning rate to specialise them to your data. The rule of thumb: **feature-extract first; fine-tune only if you need more and have enough data.**

---
## Part 10 — Recurrent networks for sequences

So far every input was a fixed, unordered bundle of numbers. But lots of data is a **sequence** where *order matters*: stock prices, sensor readings, sales over time, words in a sentence. **Recurrent Neural Networks (RNNs)** read a sequence step by step, carrying a "memory" of what they've seen — perfect for time series and text.

The original course used a real traffic-volume dataset; to keep this notebook file-free, we'll **generate a synthetic time series** (a repeating wave with an upward trend and noise) and learn to forecast the next value. The technique is identical on real data.

In [ ]:
# Build a synthetic daily-ish signal: seasonality + trend + noise
t = np.arange(2000)
series = (
      np.sin(t * 2 * np.pi / 50)  # repeating seasonal wave
      + 0.02 * t / 10  # slow upward trend
      + 0.3 * np.random.randn(len(t))
)  # noise
series = series.astype('float32')

plt.figure(figsize=(11, 3))
plt.plot(series[:300])
plt.title('Synthetic time series (first 300 steps)')
plt.show()

### Turn a series into supervised windows
RNNs learn from `(window of past values -> next value)` pairs. We slide a window of length 30 across the series: the 30 values are the input, the 31st is the target. This "windowing" step is the heart of time-series modelling.

In [ ]:
WINDOW = 30


def make_windows(seq, window=WINDOW):
      X, y = [], []
      for i in range(len(seq) - window):
            X.append(seq[i : i + window])
            y.append(seq[i + window])
      X = np.array(X)[..., np.newaxis]  # shape (samples, window, 1 feature)
      return X, np.array(y)


Xw, yw = make_windows(series)

# time series must be split by TIME, never shuffled (no peeking at the future!)
cut = int(0.8 * len(Xw))
Xtr, Xte = Xw[:cut], Xw[cut:]
ytr, yte = yw[:cut], yw[cut:]
print('X windows:', Xtr.shape, '-> (samples, window length, features)')

### Build the RNN
We use the **functional API** here (the course's later style): define an `Input`, pass it through layers, name the `outputs`. An **`LSTM`** is a smarter RNN cell that handles long sequences without the "forgetting" problems of a plain `SimpleRNN`. Since we're predicting a number, it's a regression head again: `Dense(1)`, `mse` loss.

In [ ]:
inputs = keras.Input(shape=(WINDOW, 1))
x = layers.LSTM(32)(inputs)  # reads the sequence, outputs a summary vector
outputs = layers.Dense(1)(x)  # regression: predict the next value
rnn = keras.Model(inputs, outputs)

rnn.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = rnn.fit(Xtr, ytr, validation_split=0.1, epochs=15, batch_size=32, verbose=0)
plot_history(history, metric='mae')

In [ ]:
# Forecast on the held-out tail and compare to reality
pred = rnn.predict(Xte, verbose=0).ravel()
plt.figure(figsize=(11, 3))
plt.plot(yte, label='actual')
plt.plot(pred, label='predicted')
plt.title('RNN forecast vs actual (test window)')
plt.legend()
plt.show()
print(f'Test MAE: {rnn.evaluate(Xte, yte, verbose=0)[1]:.3f}')

**Sequence lessons that carry to real data:** window your series into (past → next) pairs, split by **time** (never shuffle a time series), reach for **LSTM** or **GRU** over `SimpleRNN` for anything but the shortest sequences, and use `return_sequences=True` when stacking recurrent layers.

---
## Part 11 — Embeddings: teaching a network to read

In Part 4 we turned reviews into 10,000-long multi-hot vectors. That works, but it's wasteful and it treats every word as unrelated — "great" and "excellent" are as different as "great" and "terrible."

An **embedding** fixes this. It's a lookup table that maps each word to a short dense vector (say 32 numbers) that the network **learns**. Words used in similar ways drift to similar vectors, so the model gets a sense of *meaning* for free. Embeddings are the foundation of virtually all modern NLP.

We'll redo IMDB sentiment, but this time keep the word **order** and feed it through an `Embedding` layer.

In [ ]:
from tensorflow.keras.datasets import imdb

NUM_WORDS = 10000
MAXLEN = 200  # keep the first 200 words of each review
(tr, ytr), (te, yte) = imdb.load_data(num_words=NUM_WORDS)

# pad/truncate every review to the same length so they stack into one array
x_tr = keras.utils.pad_sequences(tr, maxlen=MAXLEN)
x_te = keras.utils.pad_sequences(te, maxlen=MAXLEN)
ytr, yte = np.asarray(ytr, 'float32'), np.asarray(yte, 'float32')
print(
      'Padded review shape:', x_tr.shape
)  # (25000, 200) -- each row is 200 word-indices

The model:
- **`Embedding`** turns each word-index into a learned 32-D vector → output is `(200 words, 32 numbers)`.
- **`GlobalAveragePooling1D`** averages those into a single 32-D "meaning of the review" vector.
- A `Dense` sigmoid finishes the familiar binary-classification head.

In [ ]:
embed_model = keras.Sequential(
      [
            layers.Input(shape=(MAXLEN,)),
            layers.Embedding(
                  input_dim=NUM_WORDS, output_dim=32
            ),  # <-- the learned word vectors
            layers.GlobalAveragePooling1D(),
            layers.Dense(16, activation='relu'),
            layers.Dense(1, activation='sigmoid'),
      ]
)
embed_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history = embed_model.fit(
      x_tr, ytr, epochs=8, batch_size=512, validation_split=0.2, verbose=0
)
plot_history(history, metric='accuracy')
print(f'Test accuracy: {embed_model.evaluate(x_te, yte, verbose=0)[1]:.3f}')

### A stronger version: read the sequence in order with an LSTM
Averaging ignores word order ("not good" vs "good"). Feeding the embeddings into an **LSTM** (from Part 10) lets the model read the review as a sequence — often a nice accuracy bump. This is exactly how text and time-series ideas combine.

In [ ]:
lstm_text = keras.Sequential(
      [
            layers.Input(shape=(MAXLEN,)),
            layers.Embedding(input_dim=NUM_WORDS, output_dim=32),
            layers.Bidirectional(
                  layers.LSTM(32)
            ),  # reads the review forwards AND backwards
            layers.Dense(1, activation='sigmoid'),
      ]
)
lstm_text.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_text.fit(x_tr, ytr, epochs=3, batch_size=512, validation_split=0.2, verbose=2)
print(f'LSTM test accuracy: {lstm_text.evaluate(x_te, yte, verbose=0)[1]:.3f}')

> 📚 **Pretrained embeddings & beyond.** Just like pretrained *image* models (Part 9), you can load pretrained *word* vectors (GloVe, word2vec) instead of learning them from scratch — a big help when your text dataset is small. And the modern **Transformer** models behind today's large language models are, at heart, a much more powerful evolution of these same embedding ideas. You've now seen the seed of how machines read.

---
## Part 12 — Cheat sheets & where to go next

You've now built models for tables, images, sequences, and text. Here's everything condensed into references you can come back to.

### The universal Keras workflow (memorise this rhythm)
```python
# 1. LOAD & LOOK       — get the data, print shapes, plot a few examples
# 2. PREP              — scale/standardise inputs; format labels; split train/val/test
# 3. BUILD             — keras.Sequential([...]) or the functional API
# 4. COMPILE           — model.compile(optimizer, loss, metrics)   <-- match to the task!
# 5. FIT               — model.fit(X, y, epochs, batch_size, validation_split, callbacks)
# 6. EVALUATE          — model.evaluate(X_test, y_test); read the curves & metrics
```

### Pick the right output layer, loss & metric
| Task | Last layer | Loss | Main metric(s) |
|---|---|---|---|
| Binary classification | `Dense(1, "sigmoid")` | `binary_crossentropy` | accuracy, precision/recall, AUC |
| Multi-class (one label) | `Dense(C, "softmax")` | `categorical_crossentropy`* | accuracy |
| Multi-label (many labels) | `Dense(C, "sigmoid")` | `binary_crossentropy` | per-label accuracy, AUC |
| Regression | `Dense(1)` (no activation) | `mse` (or `mae`) | MAE, RMSE, R² |

\* use `sparse_categorical_crossentropy` if your labels are integers instead of one-hot.

### Pick the right layers for the data
| Data looks like | Reach for |
|---|---|
| A table of numbers | `Dense` layers (an MLP) |
| Images / grids | `Conv2D` + `MaxPooling2D` (a CNN) — or transfer learning |
| Sequences / time series / text | `LSTM` / `GRU`; text starts with an `Embedding` |
| Not enough labelled data | transfer learning + data augmentation |

### When something goes wrong
| Symptom | Likely cause | Try |
|---|---|---|
| Training loss won't drop | learning rate off; bad data prep | lower/raise LR; check scaling; simplify |
| Val loss rises while train falls | **overfitting** | dropout, L2, smaller model, more data, early stopping |
| Both losses stay high | **underfitting** | bigger model, train longer, better features |
| Great val score, poor in production | **data leakage** | re-check the split; scale using train stats only |
| Loss becomes `NaN` | LR too high; unscaled inputs | drop LR; standardise inputs |

### The one big idea to take away
Every model in this notebook — the digit classifier, the house-price regressor, the CNN, the transfer-learning model, the RNN, the sentiment reader — is the **same six-step workflow** with different layers, a matching loss, and the right metric. You don't need to memorise architectures. You need to:

1. Recognise the **task** (classification? regression? sequence?).
2. Pick the **output + loss + metric** from the table above.
3. Choose **layers** that fit the data's shape.
4. Train, watch the **validation curves**, and fight overfitting.
5. Judge honestly on a **held-out test set**.

That's the whole craft. Everything more advanced — Transformers, GANs, diffusion models, LLMs — is a variation on these foundations.

### Where to go next
- Swap in **your own dataset** and run the six steps end to end.
- Rebuild the image sections with a real folder of photos using `keras.utils.image_dataset_from_directory`.
- Read the excellent (free) [Keras developer guides](https://keras.io/guides/) and *Deep Learning with Python* by François Chollet (Keras's creator) — the source much of this course draws on.

**Well done — you can now build, train, evaluate, and troubleshoot neural networks with Keras. Go make something.** 🚀